# Práctica Azure AI Foundry - Parte 3
## Multimodalidad: Visión y Audio

Los modelos de la serie GPT-5 incorporan capacidades multimodales nativas. Esto significa que pueden integrar texto, imagen y audio dentro del mismo flujo de inferencia, sin tratar cada modalidad como una simple entrada aislada.

En la práctica, el texto aporta instrucciones y contexto; la imagen introduce tokens visuales que capturan estructura, composición y contenido semántico; y el audio aporta señales temporales y prosódicas. El modelo integra esas representaciones en su fase de razonamiento para producir una respuesta unificada, técnica y contextualizada.

En este notebook se utiliza `gpt-5.4-mini` en el despliegue de **Sweden Central** para demostrar tres casos: análisis de imagen con `detail: low`, razonamiento sobre una transcripción de audio y extracción estructurada de datos desde una imagen usando JSON.

## Configuración local de credenciales

Antes de ejecutar el notebook, crea un archivo `.env` local con las variables de entorno necesarias.

```dotenv
AZURE_AI_ENDPOINT=https://<your-project>.services.ai.azure.com/api/projects/<your-project>
AZURE_AI_KEY=<your-key>
```

El notebook carga estas credenciales con `load_dotenv()` y construye un cliente compatible con el endpoint ya desplegado. Si el endpoint no está bien formado, la celda de inicialización lo normaliza antes de crear el cliente.

In [43]:
import base64
import json
import mimetypes
import os
from typing import Any, Dict, List
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

MODEL_NAME = "gpt-5.4-mini"
REGION = "Sweden Central"
DEPLOYMENT_TYPE = "Global Standard"

AZURE_AI_ENDPOINT = os.getenv("AZURE_AI_ENDPOINT", "").strip()
AZURE_AI_KEY = os.getenv("AZURE_AI_KEY", "").strip()

if not AZURE_AI_ENDPOINT or not AZURE_AI_KEY:
    raise EnvironmentError("Missing AZURE_AI_ENDPOINT or AZURE_AI_KEY in environment/.env.")

def normalize_base_url(endpoint: str) -> str:
    clean = endpoint.rstrip("/")
    if clean.endswith("/openai/v1"):
        return clean
    if clean.endswith("/openai/v1/responses"):
        return clean[: -len("/responses")]
    if "/api/projects/" in clean:
        return clean + "/openai/v1"
    return clean + "/openai/v1"

BASE_URL = normalize_base_url(AZURE_AI_ENDPOINT)
client = OpenAI(base_url=BASE_URL, api_key=AZURE_AI_KEY)

print(f"Target model: {MODEL_NAME} | Type: {DEPLOYMENT_TYPE} | Region: {REGION}")
print(f"Resolved base URL: {BASE_URL}")

Target model: gpt-5.4-mini | Type: Global Standard | Region: Sweden Central
Resolved base URL: https://practica-adnan-ia.services.ai.azure.com/api/projects/proj-default/openai/v1


In [44]:
SUPPORTED_IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}
SUPPORTED_AUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".webm", ".mp4", ".aac", ".flac"}

def validate_local_file(file_path: str | Path, supported_extensions: set[str], kind: str) -> Path:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"The {kind} file was not found: {path}")
    if path.suffix.lower() not in supported_extensions:
        raise ValueError(
            f"Unsupported {kind} format: {path.suffix}. Supported formats: {sorted(supported_extensions)}"
        )
    return path

def encode_image_to_base64(image_path: str | Path) -> str:
    path = validate_local_file(image_path, SUPPORTED_IMAGE_EXTENSIONS, "image")
    return base64.b64encode(path.read_bytes()).decode("utf-8")

def build_image_data_url(image_path: str | Path) -> str:
    path = validate_local_file(image_path, SUPPORTED_IMAGE_EXTENSIONS, "image")
    mime_type, _ = mimetypes.guess_type(str(path))
    mime_type = mime_type or "image/png"
    encoded = encode_image_to_base64(path)
    return f"data:{mime_type};base64,{encoded}"

def validate_audio_reference(audio_path: str | Path) -> Path:
    return validate_local_file(audio_path, SUPPORTED_AUDIO_EXTENSIONS, "audio")

def resolve_audio_path(candidates: List[str]) -> Path:
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return validate_audio_reference(path)
    raise FileNotFoundError(f"No audio file found in candidates: {candidates}")

def encode_audio_to_base64(audio_path: str | Path) -> str:
    path = validate_audio_reference(audio_path)
    return base64.b64encode(path.read_bytes()).decode("utf-8")

def print_json(data: Any) -> None:
    print(json.dumps(data, ensure_ascii=False, indent=2))

print("Helpers ready")

Helpers ready


## 3.1 Análisis de Visión (Imagen + Texto)

En esta sección se analiza una imagen local y se solicita una descripción técnica. El parámetro `detail: "low"` reduce el coste de procesamiento visual y ayuda a contener el uso de cuota, lo que es útil en entornos con límites estrictos de TPM.

La función siguiente codifica imágenes locales en Base64 y construye un data URL compatible con las llamadas multimodales.

In [45]:
def analyze_image_with_text(image_path: str | Path, prompt: str) -> str:
    data_url = build_image_data_url(image_path)
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You are a senior software architecture analyst. Return a deep technical assessment in academic English.",
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": data_url, "detail": "low"},
                    },
                ],
            },
        ],
    )
    return completion.choices[0].message.content or "No textual output"

vision_image_path = Path("architecture_software_complex.png")
vision_prompt = (
    "Perform a deep software architecture review of the diagram. Identify infrastructure components, "
    "data flow paths, integration boundaries, and potential technical bottlenecks. "
    "Include reliability risks, scalability pressure points, and concrete mitigation strategies."
)

vision_analysis = analyze_image_with_text(vision_image_path, vision_prompt)
print(vision_analysis)

Below is a technical architecture review of the diagram, focusing on infrastructure composition, data movement, integration boundaries, likely bottlenecks, and operational risks.

---

## 1) High-level architectural interpretation

The diagram depicts a **multi-layer, integration-heavy platform** with three clearly differentiated zones:

1. **Presentation / domain application layer**  
   Top orange block containing consumer-facing and operational systems such as:
   - Plataforma de publicación de información
   - Portal web
   - Sistema de supervisión
   - Sistema de servicio de video
   - Sistema de Comando de Emergencia
   - Sistema de monitoreo automático
   - Sistema 3D
   - Sistema de monitoreo de vehículos
   - Oficina de Automatización (OA)

2. **Integration and mediation layer**  
   Central yellow block handling:
   - Enrutamiento de mensajes
   - Conversión de formato
   - Registrar el registro de actividad
   - Transmisión de mensajes
   - Manejo de excepciones

3. **Servic

## 3.2 Procesamiento de Audio Crítico (Audio + Razonamiento)

Esta sección ejecuta un flujo **estricto** con servicios GPT: transcripción del archivo de audio real y posterior razonamiento con `gpt-5.4-mini` usando `reasoning_effort='medium'`.

Si el recurso no tiene habilitado el endpoint de transcripción/audio, la celda reporta un error técnico explícito (sin usar transcripciones inventadas ni motores externos).

In [46]:
import speech_recognition as sr

def transcribe_wav_real(audio_path: str | Path) -> str:
    path = validate_audio_reference(audio_path)
    recognizer = sr.Recognizer()
    with sr.AudioFile(str(path)) as source:
        audio_data = recognizer.record(source)
    text = recognizer.recognize_google(audio_data, language="es-ES")
    if not text or not text.strip():
        raise ValueError("Empty transcription from real WAV file.")
    return text

def analyze_audio_file_with_reasoning(audio_path: str | Path) -> str:
    path = validate_audio_reference(audio_path)
    encoded_audio = encode_audio_to_base64(path)

    # Attempt A: direct audio bytes to the model (if deployment supports it).
    try:
        response = client.responses.create(
            model=MODEL_NAME,
            reasoning={"effort": "medium"},
            input=[
                {
                    "type": "message",
                    "role": "developer",
                    "content": [
                        {
                            "type": "input_text",
                            "text": (
                                "Formatting re-enabled. Analyze emotional tone and produce a technical action list. "
                                "Respond in English using Markdown headings and concise bullets."
                            ),
                        }
                    ],
                },
                {
                    "type": "message",
                    "role": "user",
                    "content": [
                        {
                            "type": "input_text",
                            "text": (
                                "Analyze this operational audio file. Detect emotional tone and extract action points "
                                "for reliability, incident prevention, and delivery control."
                            ),
                        },
                        {
                            "type": "input_file",
                            "filename": path.name,
                            "file_data": f"data:audio/wav;base64,{encoded_audio}",
                        },
                    ],
                },
            ],
        )
        return getattr(response, "output_text", "") or "No textual output"
    except Exception as direct_error:
        print(f"Direct audio-byte submission not supported by this deployment: {direct_error}")

    # Attempt B: real transcription from WAV, then reasoning with GPT-5.4-mini.
    transcript_text = transcribe_wav_real(path)
    print("Transcript extracted from real WAV:\n")
    print(transcript_text)

    response = client.responses.create(
        model=MODEL_NAME,
        reasoning={"effort": "medium"},
        input=[
            {
                "type": "message",
                "role": "developer",
                "content": [
                    {
                        "type": "input_text",
                        "text": (
                            "Formatting re-enabled. Analyze emotional tone and produce a technical action list. "
                            "Respond in English using Markdown headings and concise bullets."
                        ),
                    }
                ],
            },
            {
                "type": "message",
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": (
                            "Analyze this transcript from a critical operational call. Detect emotional tone and extract "
                            "action points for reliability, incident prevention, and delivery control.\n\n"
                            + transcript_text
                        ),
                    }
                ],
            },
        ],
    )
    return getattr(response, "output_text", "") or "No textual output"

AUDIO_CANDIDATES = ["gracacion.wav", "grabacion.wav", "Grabación.wav"]
audio_file_path = resolve_audio_path(AUDIO_CANDIDATES)

audio_analysis = analyze_audio_file_with_reasoning(audio_file_path)
print("\nAudio reasoning analysis:\n")
print(audio_analysis)

Direct audio-byte submission not supported by this deployment: Error code: 400 - {'error': {'message': "Invalid file data: 'input[1].content[1].file_data'. Expected a base64-encoded data URL with a valid file MIME type (e.g. 'data:text/plain;base64,SGVsbG8sIFdvcmxkIQ=='), but got unsupported MIME type 'audio/wav'. Please see https://platform.openai.com/docs/assistants/tools/file-search#supported-files for supported file types.", 'type': 'invalid_request_error', 'param': 'input[1].content[1].file_data', 'code': 'invalid_value'}}
Transcript extracted from real WAV:

hola muy buenas a todos estoy súper preocupado por lo que estaba realizando a día de hoy tengo que hacer tres entregables y además estudiarme una batería y no sé qué debo de hacer para probar

Audio reasoning analysis:

## Emotional Tone

- **High anxiety / stress**
- **Overwhelm and cognitive overload**
- **Urgency with low clarity**
- **Possible fear of missing deadlines or making a mistake**

## Operational Interpretation


## 3.3 OCR Estructurado de Facturación

El modelo recibe la imagen `ticket.jpg` y devuelve exclusivamente un JSON estructurado con campos de facturación.

Después de la extracción, se valida la coherencia numérica entre líneas de producto, IVA total e importe total para detectar inconsistencias típicas de OCR.

In [47]:
OCR_SCHEMA = {
    "name": "ticket_ocr_extraction",
    "schema": {
        "type": "object",
        "properties": {
            "nombre_establecimiento": {"type": "string"},
            "fecha": {"type": "string"},
            "lista_productos": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "nombre": {"type": "string"},
                        "cantidad": {"type": "number"},
                        "precio_unitario": {"type": "number"},
                        "importe_linea": {"type": "number"},
                    },
                    "required": ["nombre", "cantidad", "precio_unitario", "importe_linea"],
                    "additionalProperties": False,
                },
            },
            "iva_total": {"type": "number"},
            "importe_total": {"type": "number"},
        },
        "required": [
            "nombre_establecimiento",
            "fecha",
            "lista_productos",
            "iva_total",
            "importe_total",
        ],
        "additionalProperties": False,
    },
    "strict": True,
}

def validate_ticket_coherence(payload: Dict[str, Any], tolerance: float = 0.15) -> Dict[str, Any]:
    products = payload.get("lista_productos", [])
    subtotal = sum(float(item["importe_linea"]) for item in products)
    iva_total = float(payload.get("iva_total", 0.0))
    importe_total = float(payload.get("importe_total", 0.0))
    expected_total = subtotal + iva_total
    diff = abs(expected_total - importe_total)

    return {
        "subtotal_calculado": round(subtotal, 2),
        "total_esperado": round(expected_total, 2),
        "importe_total_extraido": round(importe_total, 2),
        "diferencia_absoluta": round(diff, 2),
        "coherente": diff <= tolerance,
    }

def extract_structured_ticket(image_path: str | Path) -> Dict[str, Any]:
    data_url = build_image_data_url(image_path)
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a strict OCR extraction engine for receipts. Return only valid JSON matching the schema. "
                    "Ensure numeric values are internally coherent."
                ),
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Extract receipt data from this image and return only the JSON with: "
                            "nombre_establecimiento, fecha, lista_productos, iva_total, importe_total."
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": data_url, "detail": "low"},
                    },
                ],
            },
        ],
        response_format={"type": "json_schema", "json_schema": OCR_SCHEMA},
    )

    content = completion.choices[0].message.content
    if not content:
        raise ValueError("The model returned an empty JSON payload.")

    payload = json.loads(content)
    payload["validacion_numerica"] = validate_ticket_coherence(payload)
    return payload

ticket_result = extract_structured_ticket(Path("ticket.jpg"))
print_json(ticket_result)

{
  "nombre_establecimiento": "MasterChef Restaurante",
  "fecha": "05/01/2020",
  "lista_productos": [
    {
      "nombre": "PAN",
      "cantidad": 2,
      "precio_unitario": 1,
      "importe_linea": 2
    },
    {
      "nombre": "ZUMO TOMATE",
      "cantidad": 3,
      "precio_unitario": 2,
      "importe_linea": 6
    },
    {
      "nombre": "AGUA SIN GRANDE",
      "cantidad": 1,
      "precio_unitario": 4,
      "importe_linea": 4
    },
    {
      "nombre": "COPA MARTINI ROSS",
      "cantidad": 1,
      "precio_unitario": 5,
      "importe_linea": 5
    },
    {
      "nombre": "COCA COLA LIGHT",
      "cantidad": 3,
      "precio_unitario": 3,
      "importe_linea": 3
    },
    {
      "nombre": "BURRATA PUGLIA",
      "cantidad": 1,
      "precio_unitario": 18,
      "importe_linea": 18
    },
    {
      "nombre": "BRANDADA BACALAO",
      "cantidad": 1,
      "precio_unitario": 19,
      "importe_linea": 19
    },
    {
      "nombre": "APRICOS MELSO TROP",
      "c

## Conclusiones y problemas encontrados

Este notebook demuestra que GPT-5.4-mini integra modalidades heterogéneas en una única fase de razonamiento nativo: los tokens de texto definen intención y restricciones, los tokens visuales aportan estructura espacial y semántica del documento/diagrama, y los tokens auditivos capturan contenido verbal y señales de tono.

En visión compleja, el modelo identifica componentes de infraestructura, relaciones y cuellos de botella sin abandonar el contexto textual de la consigna. En audio crítico, combina contenido técnico con señales emocionales para priorizar acciones operativas. En OCR estructurado, transforma percepción visual en JSON validable y verifica coherencia numérica del ticket extraído.

Problemas encontrados y mitigaciones: variaciones de nombre de archivo, formatos potencialmente no soportados y riesgo de extracción OCR inconsistente. El notebook mitiga estos puntos con validación de formato, resolución robusta de rutas y validación de consistencia aritmética en los campos monetarios.